# **Trainer**

## **1. Dataset initialization**

As in the previous notebook. First set your dataset directory, initialize it and create the dataloaders for training.

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

# 1. Imports from your repo
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
from SCAPES.training.FlowModel_trainer import FlowTrainer
from SCAPES.models.factorization import LocalEncoder
from SCAPES.models.flow import FlowModel

# Fill this with the path to your dataset
path_to_dataset = "../../datasets/microtex/"

# Default configurations
segment_length, context_length, hop_size = 5, 5, 1

# dataset initialization
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=[
        "latent_past", "latent_present", "scale_past", "scale_present", 
        "ctx_emb_context_win", "index"
    ],
    device="cpu", # Leave dataset on CPU, trainer moves batches to GPU
    verbose=True
)

# create dataloaders for training and validation
train_split, val_split = dataset.get_splits()
train_loader = DataLoader(train_split, batch_size=16, shuffle=True, drop_last=True)
val_loader = DataLoader(val_split, batch_size=16, shuffle=False)

## **2. Model Configuration**

Pick a model size between `small`, `medium`, `large` and `extra_large`. If unsure just leave it as it is :)

In [ ]:
# Initialize Models
CONFIG_TYPE = "medium"

configs = {
    "small": {
        "d_model": 512,
        "num_layers": 6,
        "nhead": 8,
        "dim_feedforward": 2048,
        "max_seq_len": 2048,
    },
    "medium": {
        "d_model": 768,
        "num_layers": 8,
        "nhead": 12,
        "dim_feedforward": 2048,
        "max_seq_len": 2048,
    },
    "large": {
        "d_model": 1024,
        "num_layers": 12,
        "nhead": 16,
        "dim_feedforward": 2048,
        "max_seq_len": 2048,
    },
    "extra_large": {
        "d_model": 1024,
        "num_layers": 24,
        "nhead": 16,
        "dim_feedforward": 4096,
        "max_seq_len": 6000,
    }
}

# Extract config variables first
d_model            = configs[CONFIG_TYPE]["d_model"]
nhead              = configs[CONFIG_TYPE]["nhead"]
num_layers         = configs[CONFIG_TYPE]["num_layers"]
dim_feedforward    = configs[CONFIG_TYPE]["dim_feedforward"]
frame_dim          = 129
context_vector_dim = 1024
num_past_atoms     = segment_length
frames_per_atom    = 21

# Create config dictionaries for loading models later
encoder_config = {
    "in_channels": 129, 
    "hidden_dim": 256, 
    "out_channels": d_model, 
    "time_entanglement": True, 
    "temporal_compression": 1
}

flow_model_config = {
    "frame_dim": frame_dim,                  
    "context_vector_dim": context_vector_dim,        
    "num_past_atoms": num_past_atoms,              
    "frames_per_atom": frames_per_atom,
    "d_model": d_model,                    
    "nhead": nhead,                        
    "num_layers": num_layers,                   
    "dim_feedforward": dim_feedforward
}

# Set your device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the audio processor
processor_48k = EncodecProcessor(sr=48000, streamable=True, device=device)

# Initialize models using the dictionaries (using **kwargs unpacking makes it super clean!)
local_encoder = LocalEncoder(**encoder_config).to(device)
flow_model = FlowModel(**flow_model_config, device=device).to(device)

# Initialize optimizer for both models
optimizer = AdamW(list(flow_model.parameters()) + list(local_encoder.parameters()), lr=1e-4)

## **3. Train!**

You can listen to some audio reconstruction during training. In order to do so, fill the `TARGET_FILES` list with names of your files. Make sure to specify a model_path so you save your trained model.

In [ ]:
# PASS A LIST OF FILES YOU WANT TO MONITOR
TARGET_FILES = []

# model path
MODEL_SAVE_PATH = "../../models/example" # Make sure this directory exists or change it to an existing one

# Initialize the trainer
trainer = FlowTrainer(
    model=flow_model,
    local_encoder=local_encoder,
    train_loader=train_loader,
    val_loader=val_loader,
    dataset=dataset,
    processor=processor_48k,
    optimizer=optimizer,
    model_config=flow_model_config, # <--- PASSED HERE
    encoder_config=encoder_config,  # <--- PASSED HERE
    model_path=MODEL_SAVE_PATH,
    context_source="ctx",
    val_audio_files=TARGET_FILES,
    device=device,
    past_dropout=0.25
)

# Start Training
trainer.train(epochs=50, audio_val_freq=10, val_nfe=16)